# Measurement Session Log

Registra una sesión de medición eléctrica. Vincula mediciones con dispositivos y archivos HDF5.

Este notebook crea una memoria de tipo `experiment` con `subtype=session`.

In [ ]:
import os, sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '../..')))
sys.path.insert(0, str(PROJECT_ROOT))

from src import db
from src.readers.keithley_dean import list_measurements

db.init_db()
print('Database initialized')

## 1. Seleccionar HDF5 file

In [ ]:
# Listar archivos HDF5 disponibles
raw_dir = PROJECT_ROOT / 'data' / 'raw'
hdf5_files = sorted(raw_dir.glob('*.hdf5'))

print(f'Available HDF5 files ({len(hdf5_files)}):')
for f in hdf5_files[-10:]:  # Últimos 10
    print(f'  - {f.name}')

In [ ]:
# === MODIFICAR ESTOS VALORES ===
HDF5_FILENAME = '2026-10-02_test.hdf5'  # Cambiar al archivo que mediste
DATE = datetime.now().strftime('%Y-%m-%d')
MEASUREMENT_NOTES = ''
# ==============================

hdf5_path = PROJECT_ROOT / 'data' / 'raw' / HDF5_FILENAME
print(f'HDF5 file: {hdf5_path}')
print(f'File exists: {hdf5_path.exists()}')

## 2. Preview de mediciones

In [ ]:
if hdf5_path.exists():
    measurements = list_measurements(hdf5_path)
    
    transfer_count = sum(1 for m in measurements if m['mode'] == 'transfer')
    output_count = sum(1 for m in measurements if m['mode'] == 'output')
    
    print(f'Total measurements: {len(measurements)}')
    print(f'  - Transfer: {transfer_count}')
    print(f'  - Output: {output_count}')
    
    print(f'\nFirst 5 measurements:')
    for m in measurements[:5]:
        print(f'  - {m["name"]} ({m["mode"]})')
else:
    print(f'File not found: {hdf5_path}')

## 3. Crear memoria de sesión de medición

In [ ]:
# Crear memoria para esta sesión de medición
memory_id = db.create_memory(
    title=f'Measurement {DATE} - {HDF5_FILENAME.replace(".hdf5", "")}',
    type='experiment',
    subtype='session',
    date=DATE,
    description=f'Sesión de medición eléctrica con archivo {HDF5_FILENAME}',
    substrate='',  # Se puede llenar después
    process='electrical measurement',
    custom_fields={
        'hdf5_file': HDF5_FILENAME,
        'n_measurements': str(len(measurements)),
        'n_transfer': str(transfer_count),
        'n_output': str(output_count),
        'instrument': 'Keithley 2600',
        'notes': MEASUREMENT_NOTES,
    },
    references={
        'hdf5': [f'data/raw/{HDF5_FILENAME}'],
    }
)

print(f'Created measurement session with id={memory_id}')

## 4. Fotos de dispositivos medidos (opcional)

In [ ]:
# Buscar fotos recientes
photos_dir = PROJECT_ROOT / 'data' / 'photos'

if photos_dir.exists():
    photos = []
    for ext in ['*.jpg', '*.png', '*.jpeg']:
        photos.extend(photos_dir.rglob(ext))
    
    photos.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    
    print(f'Found {len(photos)} photos')
    
    # Referenciar primeras 3 fotos
    for photo in photos[:3]:
        rel_path = photo.relative_to(PROJECT_ROOT)
        db.add_reference(memory_id, 'photo', str(rel_path))
        print(f'  Added: {rel_path.name}')
else:
    print('No photos directory found')

## Resumen

In [ ]:
memory = db.get_memory(memory_id)

print(f'=== Measurement Session {memory_id} ===')
print(f'Title: {memory["title"]}')
print(f'Date: {memory["date"]}')

print(f'\nFields:')
for f in memory['fields']:
    if f['field_type'] == 'custom':
        print(f'  {f["field_name"]}: {f["field_value"]}')

print(f'\nReferences ({len(memory["references"])}):')
for r in memory['references']:
    print(f'  - {r["ref_type"]}: {r["ref_path"]}')

## 5. Próximo paso: Procesar datos

Ahora que registraste la sesión de medición, puedes procesar los datos:

- **Transfer curves**: `notebooks/02_transfer_curves/01_transfer_analysis.ipynb`
- **Output curves**: `notebooks/03_output_curves/01_output_analysis.ipynb`
- **Vth extraction**: `notebooks/05_save_demo/01_save_config_demo.ipynb`

Esos notebooks te mostrarán cómo extraer Vth, movilidad, y guardar los resultados en el sistema de memorias.